In [ ]:
!export KAGGLE_API_TOKEN=enter your api

In [ ]:
import os
os.environ["KAGGLE_API_TOKEN"] = "enter your api"

In [ ]:
from google.colab import files
files.upload()   # upload kaggle.json here

KeyboardInterrupt: 

In [ ]:
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle competitions files -c h-and-m-personalized-fashion-recommendations

Next Page Token = CfDJ8JvX8PYzZ5dKharEO0H57fE7iRKofg5ilZiqk5HeqJeCTaNA3-GN7FN21hUWeBoN4gKTr52dkesdCO2uAll7tW7OZpYBsbNxKLonHe6OvUxs8ioiwJ-Q19ynNAQJxiQSwLZsZFnm8E0yJKC9in_oui43zw
name                             size  creationDate                
-------------------------  ----------  --------------------------  
articles.csv                 36127865  2022-01-17 19:56:38.411000  
customers.csv               207135859  2022-01-17 19:56:42.498000  
images/010/0108775015.jpg      154607  2022-01-17 20:11:11.277000  
images/010/0108775044.jpg      106656  2022-01-17 20:11:11.289000  
images/010/0108775051.jpg      363376  2022-01-17 20:11:11.321000  
images/011/0110065001.jpg      148571  2022-01-17 20:03:53.266000  
images/011/0110065002.jpg       85663  2022-01-17 20:03:53.278000  
images/011/0110065011.jpg      122452  2022-01-17 20:03:53.251000  
images/011/0111565001.jpg      471967  2022-01-17 20:03:53.251000  
images/011/0111565003.jpg      291358  2022-01-17 20:03:53.266000  
images/

In [ ]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("hm_data")
TRANSACTIONS_CSV = DATA_DIR / "transactions_train.csv"
ARTICLES_CSV     = DATA_DIR / "articles.csv"

# 1. Load articles fully (it's relatively small)
articles = pd.read_csv(
    ARTICLES_CSV,
    dtype={"article_id": str}
)
print("Articles shape:", articles.shape)

# 2. Define which columns you actually need from transactions
usecols_tx = ["t_dat", "customer_id", "article_id", "price", "sales_channel_id"]

# Target size for your working dataset
TARGET_ROWS = 2_000_000          # change to 1_000_000 if RAM is still tight
CHUNKSIZE   = 500_000            # how many rows to read at a time

samples = []
total_rows = 0

# 3. Iterate over transactions in chunks
for chunk in pd.read_csv(
    TRANSACTIONS_CSV,
    dtype={"article_id": str, "customer_id": str},
    parse_dates=["t_dat"],
    usecols=usecols_tx,
    chunksize=CHUNKSIZE
):
    # Merge this chunk with articles
    chunk_merged = chunk.merge(articles, on="article_id", how="left")

    # How many more rows do we still need?
    remaining = TARGET_ROWS - total_rows
    if remaining <= 0:
        break

    # Decide what fraction to sample from this chunk
    frac = min(1.0, remaining / len(chunk_merged))

    # Sample from this chunk (random but reproducible)
    chunk_sample = chunk_merged.sample(frac=frac, random_state=42)
    samples.append(chunk_sample)
    total_rows += len(chunk_sample)

    print(f"Collected so far: {total_rows} rows")

# 4. Concatenate all sampled chunks into one DataFrame
merged_sample = pd.concat(samples, ignore_index=True)
print("Final sampled merged shape:", merged_sample.shape)

# 5. (Optional) Save to disk so you only do this once
MERGED_SAMPLE_CSV = DATA_DIR / "transactions_articles_sample.csv"
merged_sample.to_csv(MERGED_SAMPLE_CSV, index=False)
print("Saved sampled merged file to:", MERGED_SAMPLE_CSV)

Articles shape: (105542, 25)
Collected so far: 500000 rows
Collected so far: 1000000 rows
Collected so far: 1500000 rows
Collected so far: 2000000 rows
Final sampled merged shape: (2000000, 29)
Saved sampled merged file to: hm_data/transactions_articles_sample.csv


In [ ]:
from google.colab import files
files.download("hm_data/transactions_articles_sample.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>